In [1]:
%config Completer.use_jedi = False
import warnings
warnings.filterwarnings("ignore")
import datetime
import tensorflow as tf
import pandas as pd
import numpy as np
import os, sys
import time
sys.path.append(os.path.join(os.getcwd(), '../common'))
from utils import timer, save, load
pd.set_option('display.max_columns', None)

2023-01-07 14:01:39.857445: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Tutorial

- concept
    - Cerebro: BackTrader的基石，所有操作都是基于Cerebro
    - Feed: 将运行策略所需的基础数据加载到Cerebro中，一般为K线数据
    - Indicator: BackTrader自带的指标，并集成了talib中的指标，我们也可以选择继承一个Indicator实现自己的指标
    - Strategy: 交易策略。这里是整个过程中最复杂的部分，需要我们计算买入/卖出信号
    - Analyzer: 分析器，以图形的风险收益等指标对交易策略的回测结果进行分析评价
    - Order: 订单，记录了与当前订单相关的所有数据
    - Trader: 交易，记录了与当前交易相关的所有数据
    - Position: 持仓，记录了与当前持仓相关的所有数据
    - Broker: 可以理解为经纪人，整个策略的初始资金、交易费率、滑点等参数需要通过Broker进行设置
    - Observer: 观察者，对数据进行监控观察，比如资金曲线等
    - Plotting: 可视化组件

[Yahoo Data Source](https://sg.finance.yahoo.com/quote/AAPL/history?p=AAPL)

In [2]:
import backtrader as bt # 导入 Backtrader
 
# 实例化 cerebro
cerebro = bt.Cerebro()
# 打印初始资金
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
# 启动回测
cerebro.run()
# 打印回测完成后的资金
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

Starting Portfolio Value: 10000.00
Final Portfolio Value: 10000.00


In [3]:
TRADE_MONEY_DIR = os.path.join(os.getcwd(), f'../DataStore/trade_money/')

In [4]:
TRADE_MONEY_DIR

'/Users/qingbin.zhuang/Personal/StockProject/Debug/../DataStore/trade_money/'

In [5]:
df_trade_money = pd.DataFrame()
for _file in os.listdir(TRADE_MONEY_DIR):
    if not _file.endswith('pkl'):
        continue
    df_trade_money_tmp = load(f'{TRADE_MONEY_DIR}/{_file}')
    df_trade_money = pd.concat([df_trade_money, df_trade_money_tmp])

In [6]:
df_trade_money['trade_date'].min(), df_trade_money['trade_date'].max()

('20200102', '20221215')

In [7]:
df_trade_money01 = df_trade_money[(df_trade_money.ts_code == '000001.SZ')]
df_trade_money01['trade_date'] = pd.to_datetime(df_trade_money01['trade_date'])

In [8]:
df_trade_money01.head(2)

,ts_code,area,industry,list_date,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,up_limit,down_limit,turnover_rate,turnover_rate_f,volume_ratio,pe,pe_ttm,pb,ps,ps_ttm,dv_ratio,dv_ttm,total_share,float_share,free_share,total_mv,circ_mv,buy_sm_vol,buy_sm_amount,sell_sm_vol,sell_sm_amount,buy_md_vol,buy_md_amount,sell_md_vol,sell_md_amount,buy_lg_vol,buy_lg_amount,sell_lg_vol,sell_lg_amount,buy_elg_vol,buy_elg_amount,sell_elg_vol,sell_elg_amount,net_mf_vol,net_mf_amount
0,000001.SZ,深圳,银行,19910403,2022-07-18,13.25,13.44,13.21,13.39,13.24,0.15,1.1329,1153505.81,1536340.973,14.56,11.92,0.5944,1.3411,0.74,7.1512,6.6535,0.7728,1.5341,1.4951,1.3443,NaN,1.940592e+06,1.940552e+06,860088.1226,2.598452e+07,2.598399e+07,290314.0,38652.82,237985.0,31714.35,374690.0,49911.00,300986.0,40108.60,287278.0,38264.27,363162.0,48355.13,201224.0,26806.01,251373.0,33456.01,-79179.0,-10440.19
0,000001.SZ,深圳,银行,19910403,2021-08-03,17.99,18.15,17.66,17.89,18.01,-0.12,-0.6663,896948.57,1609422.575,19.81,16.21,0.4622,1.0428,0.83,12.0012,11.3782,1.1604,2.2611,2.2056,1.2186,1.0061,1.940592e+06,1.940575e+06,860111.3751,3.471719e+07,3.471689e+07,177206.0,31780.95,231348.0,41545.06,292854.0,52540.94,300370.0,53911.82,295290.0,52990.36,264725.0,47446.96,131599.0,23630.01,100505.0,18038.41,-140762.0,-25138.50


In [9]:
df_trade_money01['trade_date'].min(), df_trade_money01['trade_date'].max()

(Timestamp('2020-01-02 00:00:00'), Timestamp('2022-12-15 00:00:00'))

In [10]:
class TusharePdData(bt.feeds.PandasData):
    '''
    从Tushare读取A股票数据日线
    '''
    params = (
        ('datetime', "trade_date"),
        ('open', "open"),
        ('high', "high"),
        ('low', "low"),
        ('close', "close"),
        ('volume', "vol"),
        ('openinterest', None),
        ('change', "change")
    )
    
data = TusharePdData(dataname=df_trade_money01, 
                     fromdate=datetime.datetime.strptime("20200110", "%Y%m%d"), 
                     todate=datetime.datetime.strptime("20221215", "%Y%m%d"))

In [11]:
cerebro = bt.Cerebro()
cerebro.adddata(data)
cerebro.run()
# cerebro.plot()

In [12]:
cerebro.plot()

ImportError: Matplotlib seems to be missing. Needed for plotting support

In [16]:
class TestStrategy(bt.Strategy):
    """
    继承并构建自己的bt策略
    """

    def log(self, txt, dt=None, doprint=False):
        ''' 日志函数，用于统一输出日志格式 '''
        if doprint:
            dt = dt or self.datas[0].datetime.date(0)
            print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):

        # 初始化相关数据
        self.dataclose = self.datas[0].close
        self.order = None
        self.buyprice = None
        self.buycomm = None

        # 五日移动平均线
        self.sma5 = bt.indicators.SimpleMovingAverage(
            self.datas[0], period=10)
        # 十日移动平均线
        self.sma10 = bt.indicators.SimpleMovingAverage(
            self.datas[0], period=20)

    def notify_order(self, order):
        """
        订单状态处理

        Arguments:
            order {object} -- 订单状态
        """
        if order.status in [order.Submitted, order.Accepted]:
            # 如订单已被处理，则不用做任何事情
            return

        # 检查订单是否完成
        if order.status in [order.Completed]:
            if order.isbuy():
                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            self.bar_executed = len(self)

        # 订单因为缺少资金之类的原因被拒绝执行
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')

        # 订单状态处理完成，设为空
        self.order = None

    def notify_trade(self, trade):
        """
        交易成果
        
        Arguments:
            trade {object} -- 交易状态
        """
        if not trade.isclosed:
            return

        # 显示交易的毛利率和净利润
        self.log('OPERATION PROFIT, GROSS %.2f, NET %.2f' %
                 (trade.pnl, trade.pnlcomm), doprint=True)

    def next(self):
        ''' 下一次执行 '''

        # 记录收盘价
        self.log('Close, %.2f' % self.dataclose[0])

        # 是否正在下单，如果是的话不能提交第二次订单
        if self.order:
            return

        # 是否已经买入
        if not self.position:
            # 还没买，如果 MA5 > MA10 说明涨势，买入
            if self.sma5[0] > self.sma10[0]:
                self.order = self.buy()
        else:
            # 已经买了，如果 MA5 < MA10 ，说明跌势，卖出
            if self.sma5[0] < self.sma10[0]:
                self.order = self.sell()

    def stop(self):
        self.log(u'(金叉死叉有用吗) Ending Value %.2f' %
                 (self.broker.getvalue()), doprint=True)

In [17]:
# 初始化模型
cerebro = bt.Cerebro()

# 构建策略
strats = cerebro.addstrategy(TestStrategy)
# 每次买100股
cerebro.addsizer(bt.sizers.FixedSize, stake=100)

# 加载数据到模型中
# data = bt.feeds.GenericCSVData(
#     dataname='600519.csv',
#     fromdate=datetime.datetime(2010, 1, 1),
#     todate=datetime.datetime(2020, 4, 12),
#     dtformat='%Y%m%d',
#     datetime=2,
#     open=3,
#     high=4,
#     low=5,
#     close=6,
#     volume=10
# )

cerebro.adddata(data)

# 设定初始资金和佣金
cerebro.broker.setcash(1000000.0)
cerebro.broker.setcommission(0.005)

# 策略执行前的资金
print('启动资金: %.2f' % cerebro.broker.getvalue())

# 策略执行
cerebro.run()

启动资金: 1000000.00
2022-10-17, OPERATION PROFIT, GROSS -298.00, NET -310.91
2022-09-28, OPERATION PROFIT, GROSS -220.00, NET -233.19
2022-10-28, OPERATION PROFIT, GROSS -251.00, NET -262.88
2021-08-26, (金叉死叉有用吗) Ending Value 999561.67


In [18]:
cerebro.plot()

ImportError: cannot import name 'warnings' from 'matplotlib.dates' (/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/site-packages/matplotlib/dates.py)

## MA策略

In [88]:
import backtrader as bt

In [140]:
print(bt)

<module 'backtrader' from '/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/site-packages/backtrader/__init__.py'>


In [141]:
bt.feeds.PandasData?

In [132]:
class MAcrossover(bt.Strategy):
    # Moving average parameters
    params = (('pfast', 20), ('pslow', 50))
    
    def log(self, txt, dt = None):
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}')
        
    def __init__(self):
        self.dataclose = self.datas[0].close
        
        # Order variable will contain ongoing order details/status
        self.order = None

        # Instantiate moving averages
        self.slow_sma = bt.indicators.MovingAverageSimple(self.datas[0], 
                        period=self.params.pslow)
        self.fast_sma = bt.indicators.MovingAverageSimple(self.datas[0], 
                        period=self.params.pfast)
        
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'BUY EXECUTED, {order.executed.price:.2f}')
            elif order.issell():
                self.log(f'SELL EXECUTED, {order.executed.price:.2f}')
            self.bar_executed = len(self)
            
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')

        # Reset orders
        self.order = None
        
    def next(self):
        # Check for open orders
        if self.order:
            return

        # Check if we are in the market
        if not self.position:
            # We are not in the market, look for a signal to OPEN trades

            #If the 20 SMA is above the 50 SMA
            if self.fast_sma[0] > self.slow_sma[0] and self.fast_sma[-1] < self.slow_sma[-1]:
                self.log(f'BUY CREATE {self.dataclose[0]:2f}')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.buy()
            #Otherwise if the 20 SMA is below the 50 SMA   
            elif self.fast_sma[0] < self.slow_sma[0] and self.fast_sma[-1] > self.slow_sma[-1]:
                self.log(f'SELL CREATE {self.dataclose[0]:2f}')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()
        else:
            # We are already in the market, look for a signal to CLOSE trades
            if len(self) >= (self.bar_executed + 5):
                self.log(f'CLOSE CREATE {self.dataclose[0]:2f}')
                self.order = self.close()



In [134]:
import datetime
import backtrader as bt

cerebro = bt.Cerebro(optreturn=False)

In [135]:
data = bt.feeds.YahooFinanceCSVData(dataname='Test.csv')


In [136]:
cerebro.adddata(data)


In [137]:
#Add strategy to Cerebro
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe_ratio')
cerebro.optstrategy(MAcrossover, pfast=range(5, 20), pslow=range(50, 100))  

#Default position size
cerebro.addsizer(bt.sizers.SizerFix, stake=3)

In [139]:
cerebro.run()

[]

In [138]:
optimized_runs = cerebro.run()

final_results_list = []
for run in optimized_runs:
    for strategy in run:
        PnL = round(strategy.broker.get_value() - 10000,2)
        sharpe = strategy.analyzers.sharpe_ratio.get_analysis()
        final_results_list.append([strategy.params.pfast, 
            strategy.params.pslow, PnL, sharpe['sharperatio']])

sort_by_sharpe = sorted(final_results_list, key=lambda x: x[3], 
                         reverse=True)
for line in sort_by_sharpe[:5]:
    print(line)

Process SpawnPoolWorker-3:
Process SpawnPoolWorker-9:
Process SpawnPoolWorker-1:
Process SpawnPoolWorker-7:
Process SpawnPoolWorker-6:
Process SpawnPoolWorker-4:
Process SpawnPoolWorker-2:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/User

    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-16:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Ca

Process SpawnPoolWorker-39:
Process SpawnPoolWorker-41:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-38:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3

Process SpawnPoolWorker-50:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-49:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/mu

Process SpawnPoolWorker-62:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-66:
Process SpawnPoolWorker-61:
Process SpawnPoolWorker-63:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/

...

  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-679:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickl

  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-700:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*

AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-717:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-718:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap


  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'MAcrossover' on <module '__main__' (built-in)>
Process SpawnPoolWorker-742:
Traceback (most recent call last):
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/qingbin.zhuang/opt/anaconda3/envs/env39/lib/python3.9/multiprocessing/queues.py", line 367, in get
    return _ForkingPickl

KeyboardInterrupt: 

In [2]:
import backtrader as bt

In [3]:
class MyStrategy(bt.Strategy):
    def next(self):
        pass

In [4]:
cerebro = bt.Cerebro()

In [5]:
cerebro.addstrategy(MyStrategy)
cerebro.run()

[]

In [12]:
data = bt.feeds.YahooFinanceCSVData(
    dataname='TSLA.csv',
    fromdate=datetime.datetime(2022, 11, 1),
    todate=datetime.datetime(2022, 12, 1),
)

In [29]:
data = bt.feeds.YahooFinanceCSVData(dataname='Test.csv')


In [71]:
class PrintClose(bt.Strategy):

    def __init__(self):
        #Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.datas[0].close
        

    def log(self, txt, dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}') #Print date and close

    def next(self):
        self.log('Close: %.2f | %.2f | %.2f | %.2f |' % 
                 (self.dataclose[0], self.dataclose[1], self.dataclose[2], self.dataclose[3]))

#Instantiate Cerebro engine
cerebro = bt.Cerebro()

In [72]:
data = bt.feeds.YahooFinanceCSVData(dataname='AAPL.csv')

cerebro.adddata(data)

In [74]:
cerebro.addstrategy(PrintClose)


0

In [87]:
data[0]

140.94

In [75]:
cerebro.run()


2021-12-13 Close: 174.73 | 173.32 | 178.27 | 171.27 |
2021-12-14 Close: 173.32 | 178.27 | 171.27 | 170.15 |
2021-12-15 Close: 178.27 | 171.27 | 170.15 | 168.77 |
2021-12-16 Close: 171.27 | 170.15 | 168.77 | 171.99 |
2021-12-17 Close: 170.15 | 168.77 | 171.99 | 174.63 |
2021-12-20 Close: 168.77 | 171.99 | 174.63 | 175.26 |
2021-12-21 Close: 171.99 | 174.63 | 175.26 | 179.29 |
2021-12-22 Close: 174.63 | 175.26 | 179.29 | 178.26 |
2021-12-23 Close: 175.26 | 179.29 | 178.26 | 178.34 |
2021-12-27 Close: 179.29 | 178.26 | 178.34 | 177.17 |
2021-12-28 Close: 178.26 | 178.34 | 177.17 | 176.55 |
2021-12-29 Close: 178.34 | 177.17 | 176.55 | 180.96 |
2021-12-30 Close: 177.17 | 176.55 | 180.96 | 178.66 |
2021-12-31 Close: 176.55 | 180.96 | 178.66 | 173.91 |
2022-01-03 Close: 180.96 | 178.66 | 173.91 | 171.01 |
2022-01-04 Close: 178.66 | 173.91 | 171.01 | 171.18 |
2022-01-05 Close: 173.91 | 171.01 | 171.18 | 171.20 |
2022-01-06 Close: 171.01 | 171.18 | 171.20 | 174.07 |
2022-01-07 Close: 171.18 | 1

IndexError: array index out of range

In [25]:
from yahoo_fin.stock_info import get_data


In [65]:
amazon_weekly= get_data("amzn", start_date="10/04/2022", end_date="12/04/2022", interval="1wk")


In [66]:
amazon_weekly

,open,high,low,close,adjclose,volume,ticker
2022-10-03,113.580002,123.000000,112.449997,114.559998,114.559998,258903800,AMZN
2022-10-10,115.099998,116.250000,105.349998,106.900002,106.900002,299106000,AMZN
2022-10-17,110.110001,119.589996,110.089996,119.320000,119.320000,280043100,AMZN
2022-10-24,119.980003,121.320000,97.660004,103.410004,103.410004,522007200,AMZN
2022-10-31,103.559998,104.870003,88.040001,90.980003,90.980003,654167800,AMZN
2022-11-07,91.949997,101.190002,85.870003,100.790001,100.790001,542000700,AMZN
2022-11-14,98.769997,103.790001,92.480003,94.139999,94.139999,453874300,AMZN
2022-11-21,93.970001,95.019997,90.589996,93.410004,93.410004,241025600,AMZN
2022-11-28,93.930000,97.230003,91.440002,94.129997,94.129997,384053600,AMZN


In [28]:
amazon_weekly.to_csv("Test.csv")

In [114]:
df = pd.read_csv("/Users/qingbin.zhuang/Downloads/flink_job_3dims_gb_version_20221214-094448.csv")

In [123]:
pd.Series([1,2],index=['f','gg'])

f     1
gg    2
dtype: int64

In [124]:
def t(data):
    return (data['auc'] * data['pv']).sum() / data['pv'].sum()

def tmax(data):
    return pd.Series([data['auc'].mean(), 
        (data['auc'] * data['pv']).sum() / data['pv'].sum(), 
        data[data['pv'] == data['pv'].max()]['auc'].values[0]],
        index = ['mean', 'weight', 'max_pv'])
                      
                      
                      

In [131]:

print("datetime\tmean\tweight\tmax_pv")
for _, row in df.groupby('datetime').apply(tmax).reset_index().iterrows():
    print(f'{row["datetime"]}\t{row["mean"]}\t{row["weight"]}\t{row["max_pv"]}')
    
    

datetime	mean	weight	max_pv
2022-12-12/00	0.692599	0.692599	0.692599
2022-12-12/01	0.696047	0.696047	0.696047
2022-12-12/02	0.697488	0.697488	0.697488
2022-12-12/03	0.704054	0.704054	0.704054
2022-12-12/04	0.712393	0.712393	0.712393
2022-12-12/05	0.709528	0.709528	0.709528
2022-12-12/06	0.707647	0.707647	0.707647
2022-12-12/07	0.699978	0.6999780000000001	0.699978
2022-12-12/08	0.708651	0.708651	0.708651
2022-12-12/09	0.709947	0.7104690071778583	0.713121
2022-12-12/10	0.6181785	0.7091678670975634	0.709622
2022-12-12/11	0.712827	0.7128270000000001	0.712827
2022-12-12/12	0.713846	0.7138460000000001	0.713846
2022-12-12/13	0.71374	0.7137400000000002	0.71374
2022-12-12/14	0.717905	0.717905	0.717905
2022-12-12/15	0.716638	0.7166380000000001	0.716638
2022-12-12/16	0.71862	0.7186200000000001	0.71862
2022-12-12/17	0.71368	0.71368	0.71368
2022-12-12/18	0.7077	0.7077000000000001	0.7077
2022-12-12/19	0.713823	0.7138230000000001	0.713823
2022-12-12/20	0.714306	0.714306	0.714306
2022-12-12/21	0.71130

In [126]:
df.groupby('datetime').apply(t).reset_index().0

SyntaxError: invalid syntax (791809473.py, line 1)

In [107]:
df[df['pv'] == df['pv'].max]

,datetime,dt,hour,group_name,ranker_version,uv,pv,n_pos,n_neg,auc,uauc,wuauc


In [109]:
df['pv'].max()

2025127